In [1]:
!pip install -q google-genai underthesea transformers torch numpy qdrant-client uuid6
!pip install qdrant-client sentence-transformers -q
!pip install -q groq  pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.7 MB/s eta 0:00:00


# Mục mới

In [20]:
# str(uuid6.uuid7())

### import and config
import, danh sách can chi, check can chi trong văn bản (đầu trang hoặc đầu dòng (\n))  

Số lượng trang cho từng lần gọi api  

Các kí hiệu chọn để ngắt dòng (RecursiveCharacterTextSplitter )  

Số lượng word, min and max overlap, max context lenght của model embedding   


In [2]:

from __future__ import annotations

import abc
import json
import os
import gc
import pathlib
import re
import time
import urllib.request
import uuid6
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
from pydantic import BaseModel
from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize
from google.colab import userdata
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# 60-cycle sexagenary calendar names (Giáp Tý → Quý Hợi)
CAN_CHI_60: list[str] = [
    "Giáp Tý",  "Ất Sửu",  "Bính Dần", "Đinh Mão", "Mậu Thìn", "Kỷ Tỵ",
    "Canh Ngọ", "Tân Mùi", "Nhâm Thân","Quý Dậu",  "Giáp Tuất","Ất Hợi",
    "Bính Tý",  "Đinh Sửu","Mậu Dần",  "Kỷ Mão",   "Canh Thìn","Tân Tỵ",
    "Nhâm Ngọ", "Quý Mùi", "Giáp Thân","Ất Dậu",   "Bính Tuất","Đinh Hợi",
    "Mậu Tý",   "Kỷ Sửu",  "Canh Dần", "Tân Mão",  "Nhâm Thìn","Quý Tỵ",
    "Giáp Ngọ", "Ất Mùi",  "Bính Thân","Đinh Dậu", "Mậu Tuất", "Kỷ Hợi",
    "Canh Tý",  "Tân Sửu", "Nhâm Dần", "Quý Mão",  "Giáp Thìn","Ất Tỵ",
    "Bính Ngọ", "Đinh Mùi","Mậu Thân", "Kỷ Dậu",   "Canh Tuất","Tân Hợi",
    "Nhâm Tý",  "Quý Sửu", "Giáp Dần", "Ất Mão",   "Bính Thìn","Đinh Tỵ",
    "Mậu Ngọ",  "Kỷ Mùi",  "Canh Thân","Tân Dậu",  "Nhâm Tuất","Quý Hợi",
]


# Sort longest-first to prevent sub-string matches (e.g. "Giáp" before "Giáp Tý")
_CAN_CHI_PATTERN: str = "|".join(
    re.escape(cc) for cc in sorted(CAN_CHI_60, key=len, reverse=True)
)
print(f"in can chi nè : {_CAN_CHI_PATTERN}\n")


# Danh sách tên các tháng trong năm (Tháng 1 → Tháng 12)
MONTHS_LIST: list[str] = [
    "Tháng 1", "Tháng 2", "Tháng 3", "Tháng 4", "Tháng 5", "Tháng 6",
    "Tháng 7", "Tháng 8", "Tháng 9", "Tháng 10", "Tháng 11", "Tháng 12"
]

# Tạo Pattern mẫu dạng: (Tháng 1|Tháng 2|...|Tháng 12)
_MONTHS_PATTERN: str = "|".join(MONTHS_LIST)

# Regex nhận diện Tháng ở đầu dòng (hỗ trợ cả trường hợp có mã [54a] hoặc không)
# MONTHS_LINE_START_RE: re.Pattern = re.compile(
#     r"^(?:\[\d{2}[ab]\]\s+)?(" + _MONTHS_PATTERN + r")(?=\s*|\,|[\[\(\.]|$)",
#     re.MULTILINE
# )


# Regex hỗ trợ cả: [Mã]? + [Năm]? + [Can Chi hoặc Tháng] + [Nhìn trước]
CAN_CHI_MONTH_RE: re.Pattern = re.compile(
    r"^(?:\[\d{1,2}[ab]\]\s+)?(?:Năm\s+)?(?:" + _CAN_CHI_PATTERN + r"|" + _MONTHS_PATTERN + r")(?=\s*|\,|[\[\(\.]|$)",
    re.MULTILINE
)



# Tìm thẻ và trang trong văn bản [1], [2], …  , [PAGE:12]
FOOTNOTE_REF_RE: re.Pattern = re.compile(r"\[(\d{1,2})\]")
PAGE_MARKER_RE: re.Pattern = re.compile(r"\n\s*\[\[\[PAGE:(\d+)\]\]\]\s*\n")


# Separator priority list for word-aware text splitting
SEPARATORS: list[str] = ["\n\n", "\n", ".", "?", "!", ";", ":", ",", " ", ""]


# Sliding-window page counts per book
BOOK_WINDOW_SIZES: dict[str, int] = {
    "Đại Việt Sử Ký Toàn Thư":                3,
    "Khâm Định Việt Sử Thông Giám Cương Mục":   3,
    "Việt Sử Toàn Thư":                        3,
    "Vương Triều Trần":                         5
}
DEFAULT_WINDOW_SIZE: int = 5


# Model IDs
# GEMINI_MODEL_ID: str = "gemini-2.5-flash"
GEMINI_MODEL_ID: str = "gemini-3.1-flash-lite"
# GEMINI_MODEL_ID: str = "Gemini 3.5 Flash"

GROQ_MODEL_LLAMA_70B = "llama-3.3-70b-versatile"
GROQ_MODEL_LLAMA_8B  = "llama-3.1-8b-instant"
GROQ_MODEL_QWEN      = "qwen/qwen3-32b"


EMBEDDING_MODELS = {
    "bkai":        {"model_id": "bkai-foundation-models/vietnamese-bi-encoder"},
    "dangvantuan": {"model_id": "dangvantuan/vietnamese-embedding"},
    "bge_m3":      {"model_id": "BAAI/bge-m3"},
    "gte":         {"model_id": "Alibaba-NLP/gte-multilingual-base"},
    "halong":      {"model_id": "hiieu/halong_embedding"},
    "aiteamvn":    {"model_id": "AITeamVN/Vietnamese_Embedding"},
}

MODEL_KEY = "bkai"
# MODEL_KEY = "halong"
# MODEL_KEY = "gte"
# MODEL_KEY = "aiteamvn"
# MODEL_KEY = "bge_m3"
# MODEL_KEY = "dangvantuan"

CHUNK_CONTEXT_LENGTH = 256

EMBED_MODEL_ID = EMBEDDING_MODELS[MODEL_KEY]["model_id"]

CONTEXT_LENGTHS = [256, 512, 1024]


FILES_MAP = {
    256: [
        "full_VTT_data_qdrant_points_256.json",
        "full_VietSu_data_qdrant_points_256.json",
        "full_KhamDinh_data_qdrant_points_256.json",
        "full_DVSK_data_bkai_256.json"
    ],
    512: [
        "full_VTT_data_qdrant_points_512.json",
        "full_KhamDinh_data_qdrant_points_512.json",
        "full_VietSu_data_qdrant_points_512.json",
        "full_DVSK_data_aiteamvn_512.json"
    ],
    1024: [
        "full_VTT_data_qdrant_points_1024.json",
        "full_VietSu_data_qdrant_points_1024.json",
        "full_KhamDinh_data_qdrant_points_1024.json",
        "full_DVSK_data_aiteamvn_1024.json"
    ]
}

# GEMINI_MODEL_ID: str = "gemini-2.5-flash"
GEMINI_MODEL_ID: str = "gemini-3.1-flash-lite"

GROQ_MODEL_LLAMA_70B = "llama-3.3-70b-versatile"
GROQ_MODEL_LLAMA_8B  = "llama-3.1-8b-instant"
GROQ_MODEL_QWEN      = "qwen/qwen3-32b"



CHUNKING_CONFIGS = {
    256: {
        # semantic target
        "ideal_min_chunks": 2,
        "ideal_max_chunks": 3,

        # hard cap by mini-chunk count
        "max_group_size": 5,

        # token constraints
        "target_token_min": 90,
        "target_token_max": 210,

        # hard safety limits
        "min_chunk_tokens": 70,
        "max_chunk_tokens": 240,

        # overlap
        "overlap_tokens": 20,

        "output_file": "chunks_256.json",
    },

    512: {
        "ideal_min_chunks": 3,
        "ideal_max_chunks": 6,

        "max_group_size":9,

        "target_token_min": 180,
        "target_token_max": 400,

        # tránh sát trần 512
        "min_chunk_tokens": 120,
        "max_chunk_tokens": 460,

        "overlap_tokens": 30,

        "output_file": "chunks_512.json",
    },

    1024: {
        "ideal_min_chunks": 6,
        "ideal_max_chunks": 12,

        "max_group_size": 17,

        "target_token_min": 350,
        "target_token_max": 800,

        "min_chunk_tokens": 220,
        "max_chunk_tokens": 950,

        "overlap_tokens": 40,

        "output_file": "chunks_1024.json",
    },

    2048: {
        "ideal_min_chunks": 12,
        "ideal_max_chunks": 24,

        "max_group_size": 32,

        "target_token_min": 700,
        "target_token_max": 1500,

        "min_chunk_tokens": 400,
        "max_chunk_tokens": 1800,

        "overlap_tokens": 60,

        "output_file": "chunks_2048.json",
    },
}

CHUNK_CONFIG = CHUNKING_CONFIGS[CHUNK_CONTEXT_LENGTH]


# Mini-chunk & merge sizing
MINI_CHUNK_WORDS: int  = 35     # ≈ 55-70 BPE tokens for classical Vietnamese
OVERLAP_TOK_MIN:  int  = 20
OVERLAP_TOK_MAX:  int  = CHUNK_CONFIG["overlap_tokens"]

OVERLAP_TOKENS = CHUNK_CONFIG["overlap_tokens"]

MIN_CHUNK_TOKENS = CHUNK_CONFIG["min_chunk_tokens"]
MAX_EMBED_TOKENS = CHUNK_CONFIG["max_chunk_tokens"]

def build_collection_name(model_key: str, chunk_size: int) -> str:
    return f"history_{model_key}_chunk_{chunk_size}"


# Lấy cấu hình bảo mật từ Secrets của Colab
QDRANT_URL = userdata.get('QDRANT_URL')
API_KEY = userdata.get('QDRANT_API_KEY')

client = QdrantClient(url=QDRANT_URL, api_key=API_KEY)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[DEVICE] {device}")



COLLECTION_NAME = build_collection_name(
    MODEL_KEY,
    CHUNK_CONTEXT_LENGTH
)

tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_ID,
    trust_remote_code=True
)

embed_model = AutoModel.from_pretrained(
    EMBED_MODEL_ID,
    trust_remote_code=True
).to(device).eval()

VECTOR_SIZE = embed_model.config.hidden_size
print("MODEL:", MODEL_KEY)
print("MODEL ID:", EMBED_MODEL_ID)
print("COLLECTION:", COLLECTION_NAME)
print("VECTOR SIZE:", VECTOR_SIZE)

# Check if collection exists and its configuration matches
if client.collection_exists(collection_name=COLLECTION_NAME):
    collection_info = client.get_collection(collection_name=COLLECTION_NAME).config
    current_vector_size = collection_info.params.vectors.size

    if current_vector_size != VECTOR_SIZE:
        raise ValueError(
            f"Collection '{COLLECTION_NAME}' có vector size {current_vector_size}, "
            f"nhưng model hiện tại cần {VECTOR_SIZE}."
        )

    print(f"Collection '{COLLECTION_NAME}' đã tồn tại và khớp vector size {VECTOR_SIZE}.")
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE
        )
    )
    print(f"Đã tạo collection mới: {COLLECTION_NAME}, vector size={VECTOR_SIZE}.")


in can chi nè : Nhâm\ Thân|Giáp\ Tuất|Canh\ Thìn|Giáp\ Thân|Bính\ Tuất|Nhâm\ Thìn|Bính\ Thân|Giáp\ Thìn|Canh\ Tuất|Bính\ Thìn|Canh\ Thân|Nhâm\ Tuất|Bính\ Dần|Đinh\ Mão|Mậu\ Thìn|Canh\ Ngọ|Đinh\ Sửu|Nhâm\ Ngọ|Đinh\ Hợi|Canh\ Dần|Giáp\ Ngọ|Đinh\ Dậu|Mậu\ Tuất|Nhâm\ Dần|Bính\ Ngọ|Đinh\ Mùi|Mậu\ Thân|Giáp\ Dần|Giáp\ Tý|Tân\ Mùi|Quý\ Dậu|Bính\ Tý|Mậu\ Dần|Quý\ Mùi|Tân\ Mão|Canh\ Tý|Tân\ Sửu|Quý\ Mão|Tân\ Hợi|Nhâm\ Tý|Quý\ Sửu|Đinh\ Tỵ|Mậu\ Ngọ|Tân\ Dậu|Quý\ Hợi|Ất\ Sửu|Ất\ Hợi|Kỷ\ Mão|Tân\ Tỵ|Ất\ Dậu|Mậu\ Tý|Kỷ\ Sửu|Quý\ Tỵ|Ất\ Mùi|Kỷ\ Hợi|Kỷ\ Dậu|Ất\ Mão|Kỷ\ Mùi|Kỷ\ Tỵ|Ất\ Tỵ

[DEVICE] cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODEL: bkai
MODEL ID: bkai-foundation-models/vietnamese-bi-encoder
COLLECTION: history_bkai_chunk_256
VECTOR SIZE: 768
Collection 'history_bkai_chunk_256' đã tồn tại và khớp vector size 768.


### Data Structures
$$\text{Văn bản thô (chứa [PAGE:N])} \longrightarrow \text{PageSpan} \longrightarrow \text{MiniChunk} \longrightarrow \text{FinalChunk (Lưu Database)}$$

In [22]:
@dataclass
class PageSpan:
    """One [PAGE:N] block from source text, with the marker stripped."""
    page_no:      int   # -1 if text precedes the first marker
    start_offset: int   # char offset in the reconstructed marker-free text
    end_offset:   int
    text:         str


@dataclass
class MiniChunk:
    """Small ~35-word fragment before LLM semantic merging."""
    idx:           int
    text:          str


    start_offset:  int   # absolute offset in clean-text coordinate space
    end_offset:    int
    pages:         list[int]
    footnote_refs: list[str]   # ordered, deduplicated


@dataclass
class FinalChunk:
    """Production-ready chunk post-embedding."""
    chunk_id:       str
    book_name:      str
    pages:          list[int]
    raw_text:       str          # exact source text, never mutated
    segmented_text: str          # underthesea-segmented version (for embedding)
    footnote_refs:  list[str]
    footnotes:      dict[str, str]
    token_count:    int
    # vector:         list[float]


###  PAGE PARSER
Tách trang: Lưu vị trí đầu trang, cuối trang, nội dung trang  
Trả về văn bản sạch không có số trang  
Trả về số trang khi người dùng đưa cho một PageSpan  


In [23]:



def parse_page_spans(full_text: str) -> list[PageSpan]:
    # tìm thẻ page
    markers = list(PAGE_MARKER_RE.finditer(full_text))
    # print(f"tìm thẻ page: {markers}\n")

    spans:   list[PageSpan] = []
    cursor = 0   # con trỏ vị trí (vị trí tiếp theo của trang)

 # Nếu không thấy thì là trang -1 và toàn bộ văn bản đều ở 1 trang
    if not markers:
        spans.append(PageSpan(-1, 0, len(full_text), full_text))
        # print(f"[PARSER] No [PAGE:N] markers found — "
        #       f"entire text treated as page -1 ({len(full_text):,} chars)")
        return spans

    # xử lí văn bản trước thẻ PAGE
    pre = full_text[: markers[0].start()]
    if pre:
        spans.append(PageSpan(-1, 0, len(pre), pre))
        cursor = len(pre)   # chiều dài của văn bản trước PAGE
        # print(f"[PARSER] Pre-marker text: {len(pre):,} chars → page_no=-1")

# Vòng lặp xử lý từng trang (i số page tìm được còn m là đối tượng )
    for i, m in enumerate(markers):
        page_no   = int(m.group(1))    # lấy phần sau (chứa số trang )
        raw_start = m.end()
        raw_end   = markers[i + 1].start() if i + 1 < len(markers) else len(full_text)
        page_text = full_text[raw_start:raw_end]   # văn bản trong 1 trang

        span_start = cursor
        span_end   = cursor + len(page_text)
        spans.append(PageSpan(page_no, span_start, span_end, page_text))    # lưu các thông tin của trang
        cursor = span_end
        # print(f"Nội dung của một trang {page_text} \n\n\n\n\n\n")
    # print(f"[PARSER] {len(spans)} PageSpans | "
    #       f"pages {spans[0].page_no} → {spans[-1].page_no} | "
    #       f"clean-text total {cursor:,} chars")
    return spans



# trả về toàn bộ văn bản sạch không có page
def build_clean_text(spans: list[PageSpan]) -> str:
    return "".join(s.text for s in spans)




# Tìm xem đoạn từ ký tự [start] đến [end] đang nằm ở (những) trang sách số mấy? trả về [4, 5]
def map_pages_to_slice(
    spans: list[PageSpan], start: int, end: int
) -> list[int]:
    result: list[int] = []
    for s in spans:
        if s.end_offset   <= start:
            continue
        if s.start_offset >= end:
            break
        if s.page_no != -1:
            result.append(s.page_no)
    return sorted(set(result))


### Cắt theo can chi (window text)
Cắt theo trang mặc định sau đó mới duyệt ngược lại cắt theo can chi

In [24]:
@dataclass
class WindowResult:
    text: str
    pages: list[int]
    global_start: int
    global_end: int


def _get_window_size(book_name: str) -> int:
    for key, size in BOOK_WINDOW_SIZES.items():
        if key.lower() in book_name.lower() or book_name.lower() in key.lower():
            return size
    return DEFAULT_WINDOW_SIZE


def create_dynamic_can_chi_windows(
    spans: list[PageSpan], book_name: str
) -> list[WindowResult]:

    N = _get_window_size(book_name)
    print(f"\n[WINDOWS] book='{book_name}' | window_size={N} pages")

    windows: list[WindowResult] = []
    remainder_text: str = ""
    remainder_origin: int = 0

    real_spans = [s for s in spans if s.page_no != -1]
    if not real_spans:
        return []

    idx = 0
    total = len(spans)

    while idx < total or remainder_text.strip():

        batch = spans[idx: idx + N]
        is_last = idx + len(batch) >= total

        if not batch and not remainder_text.strip():
            break

        batch_text = "".join(s.text for s in batch) if batch else ""

        if remainder_text:
            pool_text = remainder_text + batch_text
            pool_start = remainder_origin
        else:
            pool_text = batch_text
            pool_start = batch[0].start_offset if batch else remainder_origin

        if not pool_text.strip():
            idx += len(batch)
            continue

        if is_last:
            cut_local = len(pool_text)

        else:
            last_page_text = batch[-1].text if batch else ""
            last_page_start_in_pool = len(pool_text) - len(last_page_text)

            cut_local = -1

            matches = list(CAN_CHI_MONTH_RE.finditer(last_page_text))
            if matches:
                cut_local = last_page_start_in_pool + matches[-1].start()

            if cut_local <= 0:
                para = last_page_text.rfind("\n\n")
                if para != -1:
                    cut_local = last_page_start_in_pool + para + 2

            if cut_local <= 0:
                nl = last_page_text.rfind("\n")
                if nl != -1:
                    cut_local = last_page_start_in_pool + nl + 1

            if cut_local <= 0:
                dot = last_page_text.rfind(".")
                if dot != -1:
                    cut_local = last_page_start_in_pool + dot + 1

            if cut_local <= 0:
                cut_local = len(pool_text)

        window_text = pool_text[:cut_local]
        window_len = len(window_text)

        if (
            not is_last
            and window_len < 6000
            and (idx + len(batch)) < total
        ):
            extra_batch = spans[idx + len(batch): idx + len(batch) + 1]

            if extra_batch:
                extra_text = "".join(s.text for s in extra_batch)

                pool_text += extra_text

                # Tính lại điểm cắt dựa trên trang cuối mới
                last_page_text = extra_batch[-1].text
                last_page_start_in_pool = len(pool_text) - len(last_page_text)

                cut_local = -1

                matches = list(CAN_CHI_MONTH_RE.finditer(last_page_text))

                if matches:
                    cut_local = last_page_start_in_pool + matches[-1].start()

                if cut_local <= 0:
                    para = last_page_text.rfind("\n\n")
                    if para != -1:
                        cut_local = last_page_start_in_pool + para + 2

                if cut_local <= 0:
                    nl = last_page_text.rfind("\n")
                    if nl != -1:
                        cut_local = last_page_start_in_pool + nl + 1

                if cut_local <= 0:
                    dot = last_page_text.rfind(".")
                    if dot != -1:
                        cut_local = last_page_start_in_pool + dot + 1

                if cut_local <= 0:
                    cut_local = len(pool_text)

                window_text = pool_text[:cut_local]
                window_len = len(window_text)

                batch.extend(extra_batch)



        window_start = pool_start
        window_end = pool_start + cut_local

        remainder_text = pool_text[cut_local:]
        remainder_origin = pool_start + cut_local

        idx += len(batch)

        if not window_text.strip():
            if is_last:
                break
            continue

        pages = map_pages_to_slice(spans, window_start, window_end)

        windows.append(WindowResult(
            text=window_text,
            pages=pages,
            global_start=window_start,
            global_end=window_end,
        ))

        if is_last:
            break

    return windows

In [25]:
# def run_pipeline(
#     input_json:         dict,
#     backends:           list[LLMBackend],
#     batch_orchestrator: Optional[GeminiBatchOrchestrator] = None,
# ) -> list[dict]:

#     book_name          = input_json["book_name"]
#     full_text          = input_json["full_text"]
#     page_footnotes_map = input_json["page_footnotes_map"]

# # bóc tách trang
#     spans = parse_page_spans(full_text)

# # chia sách thành các trang cho các lần gọi api
#     windows = create_dynamic_can_chi_windows(spans, book_name)

#     total_windows: int        = len(windows)

#     for wi, window in enumerate(windows):
#         print(f"\n{'─' * 72}")
#         pg_str = str(window.pages[:6]) + ("…" if len(window.pages) > 6 else "")
#         print(f"  WINDOW {wi + 1:>4}/{total_windows} | pages={pg_str} | chars={len(window.text):,} \n\n\n")

### Cấu trúc cắt
Cấu hình các thành phần có thể cắt tùy vào từng loại sách  

Cắt một chuỗi văn bản dài thành các đoạn nhỏ hơn


In [26]:


def detect_structural_boundaries(text: str, book_name: str) -> list[int]:
    bk = book_name
    offsets: set[int] = {0, len(text)}

    if "Đại Việt Sử Ký Toàn Thư" in bk:
        # Can-Chi names at the start of a line
        for m in CAN_CHI_MONTH_RE.finditer(text):
            offsets.add(m.start())

    elif "Khâm Định Việt Sử Thông Giám Cương Mục" in bk:
        # Lines that begin with "Năm <Can-Chi>"
        pat = re.compile(
            r"^Năm\s+(?:" + _CAN_CHI_PATTERN + r")(?=\s|$)", re.MULTILINE
        )
        for m in pat.finditer(text):
            offsets.add(m.start())

    elif "Việt Sử Toàn Thư" in bk:
        # Numbered section markers: "12 - " at line start
        for m in re.finditer(r"^\d+\s*-\s*", text, re.MULTILINE):
            offsets.add(m.start())
        # Paragraph breaks (soft boundaries)
        for m in re.finditer(r"\n{1,}", text):
            offsets.add(m.end())

    elif "Vương Triều Trần" in bk:
        # Soft paragraph breaks only
        for m in re.finditer(r"\n{1,}", text):
            offsets.add(m.end())

    else:
        for m in re.finditer(r"\n{1,}", text):
            offsets.add(m.end())

    result = sorted(offsets)
    print(f"      [STRUCT] {len(result)} structural boundaries | book='{book_name[:35]}'")
    return result



def split_text_by_words(
    text: str, max_words: int, separators: list[str]
) -> list[tuple[int, int, str]]:
    if not text.strip():
        return []
    text = re.sub(r'(?<=[a-zà-ỹ])[\n\;](?=[a-zà-ỹ])', ' ', text)
    if len(text.split()) <= max_words:
        return [(0, len(text), text)]

    results: list[tuple[int, int, str]] = []
    cursor = 0

    while cursor < len(text):
        remaining = text[cursor:]
        if not remaining.strip():
            break

        if len(remaining.split()) <= max_words:
            results.append((cursor, len(text), remaining))
            break

        word_count = 0
        in_word    = False
        approx_end = len(remaining)
# đếm số lượng từ
        for ci, ch in enumerate(remaining):
            is_ws = ch in (" ", "\n", "\t", "\r")
            if is_ws:
                in_word = False
            elif not in_word:
                in_word = True
                word_count += 1
                if word_count > max_words:
                    approx_end = ci
                    break
        # print(f"Số lượng từ: {in_word} \n\n\n")
        # ── Walk backward to find the best separator at or before approx_end ─
        cut_local: int = -1
        for sep in separators:
            if sep == "":
                # Empty string = hard-cut at approx_end (last resort)
                cut_local = approx_end
                break
            search_bound = min(approx_end + len(sep), len(remaining))
            idx = remaining.rfind(sep, 0, search_bound)
            if idx != -1 and (idx + len(sep)) > 0:
                cut_local = idx + len(sep)
                break

        if cut_local <= 0:
            cut_local = approx_end
        if cut_local <= 0 or cut_local > len(remaining):
            cut_local = len(remaining)

        frag = remaining[:cut_local]   # exact source slice — no mutation
        if frag.strip():
            results.append((cursor, cursor + cut_local, frag))

        cursor += cut_local

    return results

### Mini Chunk Creation (rule-based pre-segmentation)
Từ các trang sách tách tiếp thành các thành phần bỏ hơn  
Từ các thành phần trên tách tiếp thành các chunk có kích thước khoản 35 word

In [27]:


def create_mini_chunks(
    window: WindowResult, spans: list[PageSpan], book_name: str
) -> list[MiniChunk]:

    text       = window.text
    # print(f"window text\n {text} \n\n\n")
    boundaries = detect_structural_boundaries(text, book_name)
    # print(f"boundaries\n {boundaries} \n\n\n")

    mini_chunks: list[MiniChunk] = []
    idx = 0

    for bi in range(len(boundaries) - 1):
        seg_start = boundaries[bi]
        seg_end   = boundaries[bi + 1]
        seg_text  = text[seg_start:seg_end]   # exact source slice
        # print(f"các phần đoạn văn trong trang \n {seg_text} \n\n\n")
        if not seg_text.strip():
            continue

        frags = split_text_by_words(seg_text, MINI_CHUNK_WORDS, SEPARATORS)

        for (local_s, local_e, frag_text) in frags:
            if not frag_text.strip():
                continue

            # Absolute offsets in the global clean-text coordinate space
            abs_start = window.global_start + seg_start + local_s
            abs_end   = window.global_start + seg_start + local_e

            # Page mapping
            pages = map_pages_to_slice(spans, abs_start, abs_end)
            # print(f"map_pages_to_slice: \n {pages}  index {idx} ")
            # print(f"{frag_text} \n")
            # tok_count = inspect_tokens(frag_text, frag_text, tokenizer, "019e40d8-c484-791c-b758-df87ba39ecba")
            # print(f"tok_count {tok_count} \n\n\n")
            if not pages:
                pages = list(window.pages)   # fallback: window-level pages
                # print("méo hiểu kiểu gì \n\n\n")

            # Footnote refs: dict.fromkeys preserves insertion order & deduplicates
            refs = list(dict.fromkeys(FOOTNOTE_REF_RE.findall(frag_text)))
            # print(f"danh sách footer\n {refs} \n\n\n")
            mini_chunks.append(MiniChunk(
                idx=idx,
                text=frag_text,
                start_offset=abs_start,
                end_offset=abs_end,
                pages=pages,
                footnote_refs=refs,
            ))
            idx += 1

    print(f"    [MINI-CHUNK] window chars={len(text):,} → {len(mini_chunks)} mini-chunks")
    return mini_chunks

In [28]:
# def run_pipeline(
#     input_json:         dict,
#     backends:           list[LLMBackend],
# ) -> list[dict]:

#     book_name          = input_json["book_name"]
#     full_text          = input_json["full_text"]
#     page_footnotes_map = input_json["page_footnotes_map"]

# # bóc tách trang
#     spans = parse_page_spans(full_text)

# # chia sách thành các trang cho các lần gọi api
#     windows = create_dynamic_can_chi_windows(spans, book_name)

#     total_windows: int        = len(windows)

#     for wi, window in enumerate(windows):
#         print(f"\n{'─' * 72}")
#         pg_str = str(window.pages[:6]) + ("…" if len(window.pages) > 6 else "")
#         print(f"  WINDOW {wi + 1:>4}/{total_windows} | pages={pg_str} | chars={len(window.text):,} \n\n\n")

#         mini_chunks = create_mini_chunks(window, spans, book_name)
#         if not mini_chunks:
#             print("    [SKIP] No content mini-chunks produced for this window.")
#             continue



### call llm



#### promt và gán nhãn cho input,  output

Luồng chạy: tạo form cho các minichunk sau đó sử dụng llm. llm sử dungj

In [29]:
class SplitResponse(BaseModel):
    """
    Structured-output schema enforced on every LLM call (batch and sync).
    Guarantees the model can ONLY return a valid JSON object of this shape:
        {"split_after_indices": [3, 7, 12]}
    """
    split_after_indices: list[int]

def build_system_prompt(context_length: int, config: dict) -> str:
    return f"""\
You are an expert semantic chunk-boundary detector specializing in Vietnamese historical chronicles, royal annals, classical Vietnamese historical texts, and academic history documents.

YOUR TASK:
Analyze a sequence of pre-split Vietnamese historical text fragments called "mini-chunks".
Each mini-chunk is formatted as: <start_chunk_i tokens=X>text<end_chunk_i>
Your job is to determine ONLY the indices AFTER which a split must occur, so that the final macro-chunks are perfectly sized for a {context_length}-token embedding model.
STRICT OUTPUT RULES:
1. NEVER rewrite, summarize, translate, paraphrase, normalize, or correct the source text.
2. NEVER output any words, phrases, sentences, names, or quotes from the source text.
3. NEVER explain your reasoning.
4. NEVER use markdown fences.
5. OUTPUT ONLY one valid JSON object on a single line.
6. The JSON object must have exactly this shape:
{{"split_after_indices":[3,7,12]}}

INDEXING RULES:
- Indices are 0-based local mini-chunk indices.
- A split index i means: split AFTER mini-chunk i.
- Never return the final mini-chunk index.
- If no split is needed, return:
{{"split_after_indices":[]}}

VIETNAMESE HISTORICAL SEMANTIC SPLIT CRITERIA:
Insert a split when there is a clear shift in historical context, especially:
- A change in dynasty, reign era, historical year, or Can-Chi cycle.
- A transition to a new king, lord, general, envoy, scholar, official, or political actor.
- A shift to a different geographic location, battlefield, province, diplomatic mission, or administrative area.
- A transition between different event types, such as war, diplomacy, succession, taxation, law, ritual, disaster, rebellion, punishment, appointment, or royal order.
- A structural transition between narrative text, royal edict, decree, memorial, historian commentary, critique, citation, or explanatory note.
- A clear ending of one historical event before the text moves to another event.

ANTI-OVERFRAGMENTATION RULE:
Do NOT split merely because of sentence boundaries.
Do NOT split if the mini-chunks are part of the same continuous event, battle, royal discussion, decree, dialogue, campaign, or explanation.
Keep related historical context together when it remains coherent and within the token limits.
HARD RULE: NEVER split if the current macro-chunk accumulated so far contains fewer than {config["ideal_min_chunks"]} mini-chunks (or is under {config["target_token_min"]} token), unless absolutely forced by a shift in Dynasty. Prioritize grouping smaller events together into a single macro-chunk.
TOKEN SIZE TARGETS:
The embedding model context length is {context_length} tokens.
- Every mini-chunk now has a 'tokens' attribute (e.g., tokens=55).
- HARD LIMIT: Never allow the accumulated tokens of any macro-chunk to exceed {config["max_chunk_tokens"]} tokens.
- If a continuous historical topic is too long and the next chunk would push the total sum over {config["max_chunk_tokens"]} tokens, you MUST force a split immediately at the current index, even if the semantic event is not finished!
The preferred final macro-chunk range is approximately {config["target_token_min"]} to {config["target_token_max"]} tokens.
Avoid creating chunks below {config["min_chunk_tokens"]} tokens unless the text is a complete standalone historical unit.
Avoid creating chunks above {config["max_chunk_tokens"]} tokens.

MINI-CHUNK GROUPING TARGETS:
Prefer {config["ideal_min_chunks"]} to {config["ideal_max_chunks"]} mini-chunks per macro-chunk when the historical context is coherent.
No macro-chunk may contain more than {config["max_group_size"]} mini-chunks.

FORCED SPLIT RULE:
If no clear semantic boundary appears, force a split before the macro-chunk would exceed:
- {config["max_group_size"]} mini-chunks, or
- approximately {config["max_chunk_tokens"]} tokens.

PRIORITY ORDER:
1. Preserve historical semantic coherence.
2. Avoid exceeding token and mini-chunk limits.
3. Avoid very small chunks.
4. Prefer natural historical boundaries over fixed-size boundaries.

OUTPUT FORMAT:
Return EXACTLY one JSON object:
{{"split_after_indices":[...]}}
"""

_USER_TEMPLATE: str = """\
The following {n} tagged mini-chunks are a sequential sub-batch from a Vietnamese historical text.
Review the text content and indices carefully. Output the split indices according to the system instructions.

[START OF MINI-CHUNKS]
{tagged}
[END OF MINI-CHUNKS]

Your JSON response:"""


# form cho các minichunk trước khi gửi cho llm
def _format_tagged_chunks(chunks: list[MiniChunk]) -> str:
    return "\n".join(
        f"<start_chunk_{mc.idx}>{mc.text}<end_chunk_{mc.idx}>"
        for mc in chunks
    )


# Nhận câu trả lời của llm trả về mảng int chứa các vị trí cần cắt
def _parse_split_response(raw: str, n_chunks: int, max_group_size: int,) -> list[int]:
    clean = re.sub(r"^```[a-z]*\n?", "", raw.strip())
    clean = re.sub(r"\n?```$", "", clean.strip())
    clean = re.sub(r"<think>.*?</think>", "", clean, flags=re.DOTALL).strip()

    parsed = json.loads(clean)
    indices = parsed.get("split_after_indices", [])

    max_valid = n_chunks - 2

    #  Lọc split từ LLM
    splits = sorted(
        set(
            int(i) for i in indices
            if isinstance(i, (int, float)) and 0 <= int(i) <= max_valid
        )
    )

    #  Ép cứng: không group nào được dài hơn max_group_size
    forced_splits = []
    group_start = 0

    for split_idx in splits:
        while split_idx - group_start + 1 > max_group_size:
            forced_idx = group_start + int(max_group_size / 2)
            if forced_idx <= max_valid:
                forced_splits.append(forced_idx)
            group_start = forced_idx + int(max_group_size / 2)

        forced_splits.append(split_idx)
        group_start = split_idx + 1

    #  Xử lý đoạn cuối sau split cuối cùng
    last_idx = n_chunks - 1
    while last_idx - group_start + 1 > max_group_size:
        forced_idx = group_start + max_group_size - 1
        if forced_idx <= max_valid:
            forced_splits.append(forced_idx)
        group_start = forced_idx + 1

    return sorted(set(forced_splits))


#### chọn llm

In [30]:
# Lớp abstract cho gọi các llm
class LLMBackend(abc.ABC):
    @property
    @abc.abstractmethod
    def label(self) -> str:
        # in ra tên model hay nội dung nào đó để log
        pass

    @abc.abstractmethod # nhận system promt và user promt rồi trả về chuỗi nếu lỗi trả về lỗi
    def call(self, system: str, user: str) -> str:
        pass



class GroqBackend(LLMBackend):

    def __init__(self, api_key: str, model_id: str = GROQ_MODEL_LLAMA_70B):
        from groq import Groq
        self._client = Groq(api_key=api_key)
        self._model  = model_id

    @property
    def label(self) -> str:
        return f"groq/{self._model}"

    def call(self, system: str, user: str) -> str:
        completion = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            temperature=0.0,
            max_tokens=2048,
        )
        return completion.choices[0].message.content


class GeminiBackend(LLMBackend):

    def __init__(self, api_key: Optional[str] = None,
                 model_id: str = GEMINI_MODEL_ID):
        from google import genai
        from google.genai import types as _gtypes
        self._types  = _gtypes
        self._client = genai.Client(api_key=api_key) if api_key else genai.Client()
        self._model  = model_id

    @property
    def label(self) -> str:
        return f"gemini-sync/{self._model}"

    def call(self, system: str, user: str) -> str:
        resp = self._client.models.generate_content(
            model=self._model,
            contents=user,
            config=self._types.GenerateContentConfig(
                system_instruction=system,
                temperature=0.0,
                max_output_tokens=4096,
            ),
        )
        return resp.text


####  Gọi LLM để gộp các minichunk theo ngữ nghĩa

In [31]:
def llm_semantic_split(
    mini_chunks: list[MiniChunk],
    backends: list[LLMBackend],
    context_length: int,
    chunk_config: dict,
    max_retries: int = 3,
) -> list[int]:

    if len(mini_chunks) < 3:
        print(f"    [LLM-SYNC] Skipping — only {len(mini_chunks)} chunk(s).")
        return []

    tagged = _format_tagged_chunks(mini_chunks)
    # print(f"tagged {tagged} \n\n\n")
    user_msg = _USER_TEMPLATE.format(n=len(mini_chunks), tagged=tagged)
    system_prompt = build_system_prompt(context_length, chunk_config)
    # print(f"system prommpt {system_prompt} \n\n\n")

    n = len(mini_chunks)
    max_group_size = chunk_config["max_group_size"]

    for backend in backends:
        print(f"    [LLM-SYNC] Trying: {backend.label}")

        for attempt in range(1, max_retries + 1):
            raw = ""

            try:
                raw = backend.call(system_prompt, user_msg)

                print(f"llm trả lời \n{raw}\n")

                valid = _parse_split_response(
                    raw=raw,
                    n_chunks=n,
                    max_group_size=max_group_size,
                )

                print(
                    f"    [LLM-SYNC/{backend.label}] "
                    f"attempt={attempt} | chunks={n} → splits={valid}"
                )

                return valid

            except json.JSONDecodeError as exc:
                print(
                    f"    [LLM-SYNC/{backend.label}] JSON error "
                    f"(attempt {attempt}/{max_retries}): {exc} | "
                    f"raw={repr(raw[:120])}"
                )

            except Exception as exc:
                print(
                    f"    [LLM-SYNC/{backend.label}] Error "
                    f"(attempt {attempt}/{max_retries}): "
                    f"{type(exc).__name__}: {exc}"
                )

            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"    [LLM-SYNC/{backend.label}] Waiting {wait}s…")
                time.sleep(wait)

        print(f"    [LLM-SYNC/{backend.label}] Exhausted — next backend.")

    print("    [LLM-SYNC] ALL backends failed → zero splits for this window.")
    return []

### Semantic Merging
Merge theo kết quả của gemini trả về  
Và overlap các chunk

In [32]:


def _tok_count_full(text: str, tokenizer) -> int:
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))

 # group theo danh sách mà llm gợi ý
def _group_by_splits(
    chunks: list[MiniChunk], split_indices: list[int]
) -> list[list[MiniChunk]]:

    if not chunks:
        return []
    split_set = set(split_indices)
    groups:  list[list[MiniChunk]] = []
    current: list[MiniChunk]       = []
    # thêm các chunks vào current. Nếu gặp điểm cắt thì đẩy vào groups rồi làm sạch
    for mc in chunks:
        current.append(mc)
        if mc.idx in split_set:
            groups.append(current)
            current = []
    if current:
        groups.append(current)
    return groups


  #  raw_text   — văn bản gốc
  # embed_input— raw_text + a semantic overlap slice
  # groups chứa list chunks. trong chunks có các minichunks
def build_chunk_texts(
    groups: list[list[MiniChunk]], tokenizer
) -> list[tuple[str, str]]:

    results: list[tuple[str, str]] = []

    for i, group in enumerate(groups):
        raw   = "".join(mc.text for mc in group)
        embed = raw   # default: no overlap

        if i + 1 < len(groups): # lấy văn bản của chunk tiếp theo
            next_raw  = "".join(mc.text for mc in groups[i + 1])
            next_toks = tokenizer.encode(next_raw, add_special_tokens=False,
                                         truncation=False)
            n_next = len(next_toks)
            MAX = OVERLAP_TOK_MAX;
            MIN = OVERLAP_TOK_MIN;
            if(n_next <=  CHUNK_CONFIG["min_chunk_tokens"]):
                MAX = MAX * 3
                MIN = MIN * 3
            elif(n_next <=  (CHUNK_CONFIG["min_chunk_tokens"] + MAX)):
                MAX = MAX * 2
                MIN = MIN * 2

            if n_next > 0:
                for ovl in range(MAX, MIN - 1, -1):
                    # Estimate character count corresponding to `ovl` tokens
                    # using the token/char ratio of next_raw.
                    ratio    = min(ovl / n_next, 1.0)
                    char_end = max(1, int(len(next_raw) * ratio))

                    # Snap forward to the next whitespace to avoid mid-word cut
                    while (char_end < len(next_raw) and
                           next_raw[char_end] not in (" ", "\n", "\t", "\r")):
                        char_end += 1

                    ovl_slice = next_raw[:char_end]   # pure source slice
                    candidate = raw + " " + ovl_slice

                    if _tok_count_full(candidate, tokenizer) <= CHUNK_CONTEXT_LENGTH:
                        embed = candidate
                        break
                    # If MIN overlap still overflows, leave embed = raw
        embed = re.sub((r"\[(\d{1,2})\]"), ' ', embed)
        results.append((raw, embed))

    return results


 # gắn footer
def collect_group_metadata(
    groups: list[list[MiniChunk]], page_footnotes_map: dict
) -> list[dict]:
    metas: list[dict] = []

    for group in groups:
        all_pages: list[int] = []
        all_refs:  list[str] = []
        for mc in group:
            all_pages.extend(mc.pages)
            all_refs.extend(mc.footnote_refs)

        pages = sorted(set(all_pages))
        refs  = list(dict.fromkeys(all_refs))   # ordered dedup

        footnotes: dict[str, str] = {}
        for ref in refs:
            for pg in pages:
                pg_str = str(pg)
                if (pg_str in page_footnotes_map and
                        ref in page_footnotes_map[pg_str]):
                    footnotes[ref] = page_footnotes_map[pg_str][ref]
                    break   # first page defining this ref wins

        metas.append({
            "pages":          pages,
            "footnote_refs":  refs,
            "footnotes":      footnotes,
        })

    return metas



### Word Segmentation (underthesea) and tokenizer

In [33]:


def segment_underthesea(text: str) -> str:
    return word_tokenize(text, format="text")


def inspect_tokens(
    raw: str, seg: str, tokenizer, chunk_id: str
) -> int:
    ids   = tokenizer.encode(
        seg,
        add_special_tokens=True,
        truncation=True,
        max_length=CHUNK_CONTEXT_LENGTH,
    )
    count = len(ids)
    # print(count)

    print(f"      [TOKEN] id={chunk_id[:8]}… | "
          f"raw[0:60]={raw[:60].strip()!r}")
    print(f"              seg[0:60]={seg[:60].strip()!r}")
    print(f"              ids[:8]={ids[:8]} | total_tokens={count}")
    return count



### Embedding Model

In [34]:

def _mean_pool(model_output, attention_mask: torch.Tensor) -> torch.Tensor:
    """Compute attention-mask-weighted mean pool over token embeddings."""
    tok_emb  = model_output.last_hidden_state            # (B, T, D)
    mask_exp = attention_mask.unsqueeze(-1).expand(tok_emb.size()).float()
    return torch.sum(tok_emb * mask_exp, dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)


def embed_text(
    seg_text: str,
    tokenizer,
    model:  AutoModel,
    device: str,
) -> list[float]:
    enc = tokenizer(
        seg_text,
        padding=True,
        truncation=True,
        max_length=CHUNK_CONTEXT_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        out = model(**enc)

    emb = _mean_pool(out, enc["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb[0].cpu().numpy().tolist()


###  QDRANT PAYLOAD

$$\text{Chuỗi text thô} \xrightarrow{\text{Underthesea}} \text{word segmented} \xrightarrow{\text{Encoder + Pooling}} \text{Vector [768 chiều]} \xrightarrow{\text{Đóng gói}} \text{Qdrant Database}$$

In [35]:

def to_qdrant_point(chunk: FinalChunk) -> dict:
    return {
        "id":      chunk.chunk_id,
        # "vector":  chunk.vector,
        "payload": {
            "book_name":      chunk.book_name,
            "pages":          chunk.pages,
            "raw_text":       chunk.raw_text,
            "segmented_text": chunk.segmented_text,
            # "footnote_refs":  chunk.footnote_refs,
            "footnotes":      chunk.footnotes,
            "token_count":    chunk.token_count,
        },
    }



### MAIN
run_pipeline(input_json, backends)

1. Lấy tên sách, text, footnote
2. Tách full_text thành page spans
3. Gom page spans thành windows
4. Tạo mini-chunks cho từng window
5. Nếu dùng batch:
      gửi tất cả mini-chunk cho Gemini Batch để lấy split indices
   Nếu không:
      lát nữa gọi LLM từng window
6. Với từng window:
      8.1 lấy mini-chunk
      8.2 lấy split indices từ batch hoặc gọi LLM sync
      8.3 gộp mini-chunk thành semantic groups
      8.4 tạo raw_text + embed_input
      8.5 lấy metadata trang và footnote
      8.6 segment tiếng Việt bằng underthesea
      8.7 đếm token
7. Trả về danh sách point

In [36]:
def run_pipeline(
    input_json:         dict,
    backends:           list[LLMBackend],
) -> list[dict]:

    book_name          = input_json["book_name"]
    full_text          = input_json["full_text"]
    page_footnotes_map = input_json["page_footnotes_map"]

    print("\n" + "═" * 72)
    print(f"  [PHASE 1] RUN CHUNKING PIPELINE — '{book_name}'")
    print(f"  Using Context Length Target: {CHUNK_CONTEXT_LENGTH}")
    print("═" * 72)
# bóc tách trang
    spans = parse_page_spans(full_text)

# chia sách thành các trang cho các lần gọi api
    windows = create_dynamic_can_chi_windows(spans, book_name)

    all_chunks:    list[dict] = []
    total_windows: int        = len(windows)


# chia từng minichunk cho danh sách trang trên
    for wi, window in enumerate(windows):

        print(f"\n{'─' * 72}")
        pg_str = str(window.pages[:6]) + ("…" if len(window.pages) > 6 else "")
        print(f"  WINDOW {wi + 1:>4}/{total_windows} | pages={pg_str} | chars={len(window.text):,}")


        mini_chunks = create_mini_chunks(window, spans, book_name)
        if not mini_chunks:
            print("    [SKIP] No content mini-chunks produced for this window.")
            continue

        print(f"    [STEP5-SYNC] Calling LLM chain…")
        split_indices = llm_semantic_split( mini_chunks, backends, context_length=CHUNK_CONTEXT_LENGTH, chunk_config=CHUNK_CONFIG,)
        print(f"llm trả về \n {split_indices} \n\n\n")

# Merge groups + overlap + footnote metadata
        groups       = _group_by_splits(mini_chunks, split_indices)
        chunk_texts  = build_chunk_texts(groups, tokenizer)    # (raw, embed_input)
        metas        = collect_group_metadata(groups, page_footnotes_map)

        print(f"    [MERGE] {len(mini_chunks)} mini-chunks + {len(split_indices)} splits "
              f"→ {len(groups)} semantic groups")

        points_to_upload = []
# Segment, inspect, embed, format
        for gi, ((raw_text, embed_input), meta) in enumerate(zip(chunk_texts, metas)):
            if not raw_text.strip():
                continue

            chunk_id = str(uuid6.uuid7())

      # Word segmentation (applied to embed_input which includes overlap)
            seg_text = segment_underthesea(embed_input)

      #  Token inspection + count
            tok_count = inspect_tokens(raw_text, seg_text, tokenizer, chunk_id)

            chunk_dict = {
                "chunk_id": chunk_id,
                "book_name": book_name,
                "pages": meta["pages"],
                "raw_text": raw_text,
                "embed_input": embed_input,
                "segmented_text": seg_text,
                # "footnote_refs": meta["footnote_refs"],
                "footnotes": meta["footnotes"],
                "token_count": tok_count
            }
            all_chunks.append(chunk_dict)

        time.sleep(2.0)
    return all_chunks

#  run chunking Đại Việt Sử Ký

In [39]:
# @title

import os
from google.colab import userdata


JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư
# JSON_PATH = "full_KhamDinh_data.json"        # Khâm Định Việt Sử Thông Giám Cương Mục
# JSON_PATH = "full_VietSu_data.json"                 # Việt Sử Toàn Thư
# JSON_PATH = "full_VTT_data.json"                  # Vương Triều Trần
# JSON_PATH = "full_test.json"
# JSON_PATH = "full_test_VTT.json"


OUTPUT_PATH = JSON_PATH.replace(".json", f"_{MODEL_KEY}_{CHUNK_CONTEXT_LENGTH}.json")


def _load_key(colab_secret: str, env_var: str) -> str:
    try:
        from google.colab import userdata as _ud
        val = _ud.get(colab_secret)
        if val:
            print(f"[KEY] '{colab_secret}' loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    val = os.environ.get(env_var, "")
    if val:
        print(f"[KEY] '{env_var}' loaded from environment variable.")
    return val

GROQ_API_KEY   = _load_key("grok_key",  "grok_key")
GEMINI_API_KEY = _load_key("GEMINI_API_KEY", "GEMINI_API_KEY")

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("[KEY] ⚠  No API keys found. "
          "Add GROQ_API_KEY and/or GOOGLE_API_KEY to Colab Secrets.")


LLM_BACKENDS:       list[LLMBackend]                  = []

# if GROQ_API_KEY:
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_70B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_8B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_QWEN))

if GEMINI_API_KEY:
    try:
        LLM_BACKENDS.append(GeminiBackend(GEMINI_API_KEY, GEMINI_MODEL_ID))
        print(f"[INIT] GeminiBackend added: {GEMINI_MODEL_ID}")
    except Exception as exc:
        print(f"[INIT] GeminiBackend skipped ({type(exc).__name__}: {exc})")


if not LLM_BACKENDS:
    raise RuntimeError(
        "LLM_BACKENDS is empty. Provide GROQ_API_KEY and/or GEMINI_API_KEY."
    )


print(f"[INIT] Sync LLM chain ready ({len(LLM_BACKENDS)} backend(s)):")
for i, backend in enumerate(LLM_BACKENDS, start=1):
    print(f"   [{i}] {backend.label}")

print()

# Load input JSON
print(f"[LOAD] Reading '{JSON_PATH}'…")

with open(JSON_PATH, "r", encoding="utf-8") as fh:
    input_data: dict = json.load(fh)


assert "book_name" in input_data, "Missing key: book_name"
assert "full_text" in input_data, "Missing key: full_text"
assert "page_footnotes_map" in input_data, "Missing key: page_footnotes_map"

assert isinstance(input_data["book_name"], str), "book_name must be str"
assert isinstance(input_data["full_text"], str), "full_text must be str"
assert isinstance(input_data["page_footnotes_map"], dict), "page_footnotes_map must be dict"

print("[LOAD] Input validation passed ✓")


# Run pipeline
qdrant_points: list[dict] = run_pipeline(
    input_data,
    backends=LLM_BACKENDS,
)


# Save local output
print(f"\n[SAVE] Writing {len(qdrant_points):,} points → '{OUTPUT_PATH}'…")

with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    json.dump(qdrant_points, fh, ensure_ascii=False, indent=2)

kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"[SAVE] Done. {kb:.1f} KB written.")

[KEY] 'grok_key' loaded from Colab Secrets.
[KEY] 'GEMINI_API_KEY' loaded from Colab Secrets.
[INIT] GeminiBackend added: gemini-3.1-flash-lite
[INIT] Sync LLM chain ready (1 backend(s)):
   [1] gemini-sync/gemini-3.1-flash-lite

[LOAD] Reading 'full_DVSK_data.json'…
[LOAD] Input validation passed ✓

════════════════════════════════════════════════════════════════════════
  [PHASE 1] RUN CHUNKING PIPELINE — 'Đại Việt Sử Ký Toàn Thư'
  Using Context Length Target: 1024
════════════════════════════════════════════════════════════════════════

[WINDOWS] book='Đại Việt Sử Ký Toàn Thư' | window_size=3 pages

────────────────────────────────────────────────────────────────────────
  WINDOW    1/50 | pages=[154, 155, 156] | chars=7,436
      [STRUCT] 14 structural boundaries | book='Đại Việt Sử Ký Toàn Thư'
    [MINI-CHUNK] window chars=7,436 → 69 mini-chunks
    [STEP5-SYNC] Calling LLM chain…
    [LLM-SYNC] Trying: gemini-sync/gemini-3.1-flash-lite
llm trả lời 
{"split_after_indices":[8,16,

In [40]:
# with open("full_test_bkai_256.json", "r", encoding="utf-8") as fh:
#     loaded_chunks = json.load(fh)
# for chunk in loaded_chunks:
#     print(chunk["token_count"])
# # print(loaded_chunks)
#     # tok_count = inspect_tokens(chunk["raw_text"], chunk["embed_input"], tokenizer, chunk["chunk_id"])
#     # print (tok_count)
#     # print("\n\n")

# Run chunking Khâm Định Việt Sử Thông Giám Cương Mục

In [41]:
# @title

import os
from google.colab import userdata


# JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư
JSON_PATH = "full_KhamDinh_data.json"        # Khâm Định Việt Sử Thông Giám Cương Mục
# JSON_PATH = "full_VietSu_data.json"                 # Việt Sử Toàn Thư
# JSON_PATH = "full_VTT_data.json"                  # Vương Triều Trần
# JSON_PATH = "full_test.json"
# JSON_PATH = "full_test_VTT.json"


OUTPUT_PATH = JSON_PATH.replace(".json", f"_qdrant_points_{CHUNK_CONTEXT_LENGTH}.json")


def _load_key(colab_secret: str, env_var: str) -> str:
    try:
        from google.colab import userdata as _ud
        val = _ud.get(colab_secret)
        if val:
            print(f"[KEY] '{colab_secret}' loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    val = os.environ.get(env_var, "")
    if val:
        print(f"[KEY] '{env_var}' loaded from environment variable.")
    return val

GROQ_API_KEY   = _load_key("grok_key",  "grok_key")
GEMINI_API_KEY = _load_key("GEMINI_API_KEY", "GEMINI_API_KEY")

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("[KEY] ⚠  No API keys found. "
          "Add GROQ_API_KEY and/or GOOGLE_API_KEY to Colab Secrets.")


LLM_BACKENDS:       list[LLMBackend]                  = []

# if GROQ_API_KEY:
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_70B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_8B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_QWEN))

if GEMINI_API_KEY:
    try:
        LLM_BACKENDS.append(GeminiBackend(GEMINI_API_KEY, GEMINI_MODEL_ID))
        print(f"[INIT] GeminiBackend added: {GEMINI_MODEL_ID}")
    except Exception as exc:
        print(f"[INIT] GeminiBackend skipped ({type(exc).__name__}: {exc})")


if not LLM_BACKENDS:
    raise RuntimeError(
        "LLM_BACKENDS is empty. Provide GROQ_API_KEY and/or GEMINI_API_KEY."
    )


print(f"[INIT] Sync LLM chain ready ({len(LLM_BACKENDS)} backend(s)):")
for i, backend in enumerate(LLM_BACKENDS, start=1):
    print(f"   [{i}] {backend.label}")

print()

# Load input JSON
print(f"[LOAD] Reading '{JSON_PATH}'…")

with open(JSON_PATH, "r", encoding="utf-8") as fh:
    input_data: dict = json.load(fh)


assert "book_name" in input_data, "Missing key: book_name"
assert "full_text" in input_data, "Missing key: full_text"
assert "page_footnotes_map" in input_data, "Missing key: page_footnotes_map"

assert isinstance(input_data["book_name"], str), "book_name must be str"
assert isinstance(input_data["full_text"], str), "full_text must be str"
assert isinstance(input_data["page_footnotes_map"], dict), "page_footnotes_map must be dict"

print("[LOAD] Input validation passed ✓")


# Run pipeline
qdrant_points: list[dict] = run_pipeline(
    input_data,
    backends=LLM_BACKENDS,
)


# Save local output
print(f"\n[SAVE] Writing {len(qdrant_points):,} points → '{OUTPUT_PATH}'…")

with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    json.dump(qdrant_points, fh, ensure_ascii=False, indent=2)

kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"[SAVE] Done. {kb:.1f} KB written.")

[KEY] 'grok_key' loaded from Colab Secrets.
[KEY] 'GEMINI_API_KEY' loaded from Colab Secrets.
[INIT] GeminiBackend added: gemini-3.1-flash-lite
[INIT] Sync LLM chain ready (1 backend(s)):
   [1] gemini-sync/gemini-3.1-flash-lite

[LOAD] Reading 'full_KhamDinh_data.json'…
[LOAD] Input validation passed ✓

════════════════════════════════════════════════════════════════════════
  [PHASE 1] RUN CHUNKING PIPELINE — 'Khâm Định Việt Sử Thông Giám Cương Mục'
  Using Context Length Target: 1024
════════════════════════════════════════════════════════════════════════

[WINDOWS] book='Khâm Định Việt Sử Thông Giám Cương Mục' | window_size=3 pages

────────────────────────────────────────────────────────────────────────
  WINDOW    1/54 | pages=[187, 188, 189] | chars=7,089
      [STRUCT] 2 structural boundaries | book='Khâm Định Việt Sử Thông Giám Cương '
    [MINI-CHUNK] window chars=7,089 → 63 mini-chunks
    [STEP5-SYNC] Calling LLM chain…
    [LLM-SYNC] Trying: gemini-sync/gemini-3.1-flash-li

# Run chunking Việt Sử Toàn Thư

In [42]:
# @title

import os
from google.colab import userdata


# JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư
# JSON_PATH = "full_KhamDinh_data.json"        # Khâm Định Việt Sử Thông Giám Cương Mục
JSON_PATH = "full_VietSu_data.json"                 # Việt Sử Toàn Thư
# JSON_PATH = "full_VTT_data.json"                  # Vương Triều Trần
# JSON_PATH = "full_test.json"
# JSON_PATH = "full_test_VTT.json"


OUTPUT_PATH = JSON_PATH.replace(".json", f"_qdrant_points_{CHUNK_CONTEXT_LENGTH}.json")


def _load_key(colab_secret: str, env_var: str) -> str:
    try:
        from google.colab import userdata as _ud
        val = _ud.get(colab_secret)
        if val:
            print(f"[KEY] '{colab_secret}' loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    val = os.environ.get(env_var, "")
    if val:
        print(f"[KEY] '{env_var}' loaded from environment variable.")
    return val

GROQ_API_KEY   = _load_key("grok_key",  "grok_key")
GEMINI_API_KEY = _load_key("GEMINI_API_KEY", "GEMINI_API_KEY")

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("[KEY] ⚠  No API keys found. "
          "Add GROQ_API_KEY and/or GOOGLE_API_KEY to Colab Secrets.")


LLM_BACKENDS:       list[LLMBackend]                  = []

# if GROQ_API_KEY:
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_70B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_8B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_QWEN))

if GEMINI_API_KEY:
    try:
        LLM_BACKENDS.append(GeminiBackend(GEMINI_API_KEY, GEMINI_MODEL_ID))
        print(f"[INIT] GeminiBackend added: {GEMINI_MODEL_ID}")
    except Exception as exc:
        print(f"[INIT] GeminiBackend skipped ({type(exc).__name__}: {exc})")


if not LLM_BACKENDS:
    raise RuntimeError(
        "LLM_BACKENDS is empty. Provide GROQ_API_KEY and/or GEMINI_API_KEY."
    )


print(f"[INIT] Sync LLM chain ready ({len(LLM_BACKENDS)} backend(s)):")
for i, backend in enumerate(LLM_BACKENDS, start=1):
    print(f"   [{i}] {backend.label}")

print()

# Load input JSON
print(f"[LOAD] Reading '{JSON_PATH}'…")

with open(JSON_PATH, "r", encoding="utf-8") as fh:
    input_data: dict = json.load(fh)


assert "book_name" in input_data, "Missing key: book_name"
assert "full_text" in input_data, "Missing key: full_text"
assert "page_footnotes_map" in input_data, "Missing key: page_footnotes_map"

assert isinstance(input_data["book_name"], str), "book_name must be str"
assert isinstance(input_data["full_text"], str), "full_text must be str"
assert isinstance(input_data["page_footnotes_map"], dict), "page_footnotes_map must be dict"

print("[LOAD] Input validation passed ✓")


# Run pipeline
qdrant_points: list[dict] = run_pipeline(
    input_data,
    backends=LLM_BACKENDS,
)


# Save local output
print(f"\n[SAVE] Writing {len(qdrant_points):,} points → '{OUTPUT_PATH}'…")

with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    json.dump(qdrant_points, fh, ensure_ascii=False, indent=2)

kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"[SAVE] Done. {kb:.1f} KB written.")

[KEY] 'grok_key' loaded from Colab Secrets.
[KEY] 'GEMINI_API_KEY' loaded from Colab Secrets.
[INIT] GeminiBackend added: gemini-3.1-flash-lite
[INIT] Sync LLM chain ready (1 backend(s)):
   [1] gemini-sync/gemini-3.1-flash-lite

[LOAD] Reading 'full_VietSu_data.json'…
[LOAD] Input validation passed ✓

════════════════════════════════════════════════════════════════════════
  [PHASE 1] RUN CHUNKING PIPELINE — 'Việt Sử Toàn Thư'
  Using Context Length Target: 1024
════════════════════════════════════════════════════════════════════════

[WINDOWS] book='Việt Sử Toàn Thư' | window_size=3 pages

────────────────────────────────────────────────────────────────────────
  WINDOW    1/28 | pages=[161, 162, 163] | chars=6,972
      [STRUCT] 19 structural boundaries | book='Việt Sử Toàn Thư'
    [MINI-CHUNK] window chars=6,972 → 65 mini-chunks
    [STEP5-SYNC] Calling LLM chain…
    [LLM-SYNC] Trying: gemini-sync/gemini-3.1-flash-lite
llm trả lời 
{"split_after_indices":[2,14,25,31,49,58]}

    

# Run chunking Vương Trều Trần

In [43]:
# @title

import os
from google.colab import userdata


# JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư
# JSON_PATH = "full_KhamDinh_data.json"        # Khâm Định Việt Sử Thông Giám Cương Mục
# JSON_PATH = "full_VietSu_data.json"                 # Việt Sử Toàn Thư
JSON_PATH = "full_VTT_data.json"                  # Vương Triều Trần
# JSON_PATH = "full_test.json"
# JSON_PATH = "full_test_VTT.json"


OUTPUT_PATH = JSON_PATH.replace(".json", f"_qdrant_points_{CHUNK_CONTEXT_LENGTH}.json")


def _load_key(colab_secret: str, env_var: str) -> str:
    try:
        from google.colab import userdata as _ud
        val = _ud.get(colab_secret)
        if val:
            print(f"[KEY] '{colab_secret}' loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    val = os.environ.get(env_var, "")
    if val:
        print(f"[KEY] '{env_var}' loaded from environment variable.")
    return val

GROQ_API_KEY   = _load_key("grok_key",  "grok_key")
GEMINI_API_KEY = _load_key("GEMINI_API_KEY", "GEMINI_API_KEY")

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("[KEY] ⚠  No API keys found. "
          "Add GROQ_API_KEY and/or GOOGLE_API_KEY to Colab Secrets.")


LLM_BACKENDS:       list[LLMBackend]                  = []

# if GROQ_API_KEY:
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_70B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_8B))
#     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_QWEN))

if GEMINI_API_KEY:
    try:
        LLM_BACKENDS.append(GeminiBackend(GEMINI_API_KEY, GEMINI_MODEL_ID))
        print(f"[INIT] GeminiBackend added: {GEMINI_MODEL_ID}")
    except Exception as exc:
        print(f"[INIT] GeminiBackend skipped ({type(exc).__name__}: {exc})")


if not LLM_BACKENDS:
    raise RuntimeError(
        "LLM_BACKENDS is empty. Provide GROQ_API_KEY and/or GEMINI_API_KEY."
    )


print(f"[INIT] Sync LLM chain ready ({len(LLM_BACKENDS)} backend(s)):")
for i, backend in enumerate(LLM_BACKENDS, start=1):
    print(f"   [{i}] {backend.label}")

print()

# Load input JSON
print(f"[LOAD] Reading '{JSON_PATH}'…")

with open(JSON_PATH, "r", encoding="utf-8") as fh:
    input_data: dict = json.load(fh)


assert "book_name" in input_data, "Missing key: book_name"
assert "full_text" in input_data, "Missing key: full_text"
assert "page_footnotes_map" in input_data, "Missing key: page_footnotes_map"

assert isinstance(input_data["book_name"], str), "book_name must be str"
assert isinstance(input_data["full_text"], str), "full_text must be str"
assert isinstance(input_data["page_footnotes_map"], dict), "page_footnotes_map must be dict"

print("[LOAD] Input validation passed ✓")


# Run pipeline
qdrant_points: list[dict] = run_pipeline(
    input_data,
    backends=LLM_BACKENDS,
)


# Save local output
print(f"\n[SAVE] Writing {len(qdrant_points):,} points → '{OUTPUT_PATH}'…")

with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    json.dump(qdrant_points, fh, ensure_ascii=False, indent=2)

kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"[SAVE] Done. {kb:.1f} KB written.")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
      [TOKEN] id=019e5758… | raw[0:60]='Cùng với việc mở rộng lãnh thổ phía Nam, vua Trần Anh Tôngcò'
              seg[0:60]='Cùng với việc mở_rộng lãnh_thổ phía Nam , vua Trần_Anh_Tôngc'
              ids[:8]=[0, 126359, 1116, 2735, 20142, 454, 42, 27350] | total_tokens=504
      [TOKEN] id=019e5758… | raw[0:60]='thì ban cho một cách đầy đặn không kể đến việc đáp lại đơn'
              seg[0:60]='thì ban cho một_cách đầy_đặn không kể đến việc đáp lại đơn_s'
              ids[:8]=[0, 2579, 4599, 681, 889, 454, 238, 5687] | total_tokens=661
      [TOKEN] id=019e5758… | raw[0:60]='IH. TRẦẤN MINH TÔNG (1314 - 1329) 1, Chân dung một hoàng đế'
              seg[0:60]='IH. TRẦẤN_MINH_TÔNG ( 1314 - 1329 ) 1 , Chân_dung một hoàng_'
              ids[:8]=[0, 6, 38624, 5, 14107, 240532, 249975, 839] | total_tokens=459
      [TOKEN] id=019e5758… | raw[0:60]='Trần Minh Tông làm vua 15 năm, trải qua hai niên hiệu là Đạ'
              seg[0:6

 # Các file chunking sau khi chạy xong
 full_VTT_data_qdrant_points_1024.json  
 full_VietSu_data_qdrant_points_1024.json  
 full_KhamDinh_data_qdrant_points_1024.json  
 full_DVSK_data_aiteamvn_1024.json  
 full_VTT_data_qdrant_points_512.json  
 full_KhamDinh_data_qdrant_points_512.json  
 full_VietSu_data_qdrant_points_512.json  
 full_DVSK_data_aiteamvn_512.json  
 full_VTT_data_qdrant_points_256.json  
 full_VietSu_data_qdrant_points_256.json  
 full_KhamDinh_data_qdrant_points_256.json  
 full_DVSK_data_bkai_256.json  
